In [4]:
INPUT_FILE = "test.jsonl"
with open(INPUT_FILE, "r") as f:
    records = f.readlines()

In [12]:
len(records)

79962

In [6]:
import difflib
import re

import difflib
import re

def extract_diff_numbers(pos, neg):
    pos_tokens = pos.split()
    neg_tokens = neg.split()

    matcher = difflib.SequenceMatcher(None, pos_tokens, neg_tokens)

    pos_num = None
    neg_num = None

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == "replace":
            # Look for number in replaced segments
            for token in pos_tokens[i1:i2]:
                match = re.search(r'\d+\.\d+|\d+', token)
                if match:
                    pos_num = match.group()

            for token in neg_tokens[j1:j2]:
                match = re.search(r'\d+\.\d+|\d+', token)
                if match:
                    neg_num = match.group()

    return pos_num, neg_num


In [7]:
tmp_rec = eval(records[13])
pos_number,neg_number = extract_diff_numbers(tmp_rec['positive'], tmp_rec['negative'])
print("----")
print(tmp_rec['number'],pos_number,neg_number)


----
3 2.69 1002.21


In [8]:
import random

def generate_bounded_negative(anchor, positive, min_ratio=1.5, max_ratio=3.0):
    a = float(anchor)
    p = float(positive)

    pos_dist = round(abs(a - p), 2)

    if pos_dist == 0:
        pos_dist = max(0.01, abs(a) * 0.01)

    ratio = random.uniform(min_ratio, max_ratio)
    neg_dist = pos_dist * ratio

    # Choose direction opposite to positive
    if p > a:
        new_neg = a - neg_dist
        if new_neg < 0:
            new_neg = a + neg_dist
    else:
        new_neg = a + neg_dist

    return round(new_neg, 2)

In [9]:
def replace_number_in_text(text, old_number, new_number):
    return text.replace(old_number, str(new_number), 1)

In [61]:
import json

new_records = []
df_records = []
INPUT_FILE = "val.jsonl"

OUTPUT_FILE = "val_extracted.jsonl"
# read records 
with open(INPUT_FILE, "r") as f:
    records = f.readlines()

# extract numbers and write new records
for idx,record in enumerate(records):
    rec = eval(record)
    anchor_number = rec['number']

    pos_number, neg_number = extract_diff_numbers(rec['positive'], rec['negative'])

    if anchor_number == 0 or anchor_number == "0" or anchor_number == "0.0":
        new_pos_number = random.randint(1, 4000)
        new_neg_number = new_pos_number + random.randint(1, 100)
        rec['positive'] = replace_number_in_text(
            rec['positive'],
            pos_number,
            str(new_pos_number)
        )
        rec['positive_rewritten'] = replace_number_in_text(
            rec['positive_rewritten'],
            pos_number,
            str(new_pos_number)
        )
        rec['negative'] = replace_number_in_text(
            rec['negative'],
            neg_number,
            str(new_neg_number)
        )
        rec['negative_rewritten'] = replace_number_in_text(
            rec['negative_rewritten'],
            neg_number,
            str(new_neg_number)
        )
        rec['positive_number'] = new_pos_number
        rec['negative_number'] = new_neg_number
    
    else:    
    
        try:
            new_neg_number = generate_bounded_negative(anchor_number, pos_number)
        except Exception as e:
            print(f"idx:{idx} : {anchor_number}, pos {pos_number}, neg {neg_number}: {e}")
            continue

        rec['negative'] = replace_number_in_text(
            rec['negative'],
            neg_number,
            str(new_neg_number)
        )
        rec['negative_rewritten'] = replace_number_in_text(
            rec['negative_rewritten'],
            neg_number,
            str(new_neg_number)
        )
        _, neg_number2 = extract_diff_numbers(rec['positive'], rec['negative'])
        if neg_number2 != str(new_neg_number):
            print("****")
            print(f"idx:{idx} : {anchor_number}, pos {pos_number}, neg {neg_number} -> {neg_number2}")
            print('negative',rec['negative'], new_neg_number)
            print("****")
            continue

        rec['positive_number'] = pos_number
        rec['negative_number'] = new_neg_number
    new_records.append(rec)
    df_records.append(rec)

import jsonlines

with jsonlines.open(OUTPUT_FILE, mode='w') as writer:
    writer.write_all(new_records)



idx:106 : 0.03, pos None, neg None: float() argument must be a string or a real number, not 'NoneType'
idx:207 : 0.01, pos None, neg None: float() argument must be a string or a real number, not 'NoneType'
idx:1550 : 0.01, pos None, neg None: float() argument must be a string or a real number, not 'NoneType'
idx:1676 : 0.01, pos None, neg None: float() argument must be a string or a real number, not 'NoneType'
idx:1896 : 0.01, pos None, neg None: float() argument must be a string or a real number, not 'NoneType'
idx:2147 : 0.02, pos None, neg None: float() argument must be a string or a real number, not 'NoneType'
idx:2179 : 0.02, pos None, neg None: float() argument must be a string or a real number, not 'NoneType'
idx:2455 : 0.03, pos None, neg None: float() argument must be a string or a real number, not 'NoneType'
idx:2664 : 0.01, pos None, neg None: float() argument must be a string or a real number, not 'NoneType'
idx:3119 : 0.01, pos None, neg None: float() argument must be a st

In [62]:
len(new_records),len(records)

(9963, 9995)

In [15]:
tmp_rec = eval(records[1169])
print(tmp_rec['anchor'])
print(tmp_rec['positive'])
print(tmp_rec['negative'])


SHH Margin Trading -- Wuhan Jianmin <0.32.SS> Margin Long Amount is 1,493   (x10000), Short Selling Volume is 0.32 (x10000) yuan
SHH Margin Trading -- Wuhan Jianmin <0.36.SS> Margin Long Amount is 1,493   (x10000), Short Selling Volume is 0.32 (x10000) yuan
SHH Margin Trading -- Wuhan Jianmin <0.18.SS> Margin Long Amount is 1,493   (x10000), Short Selling Volume is 0.32 (x10000) yuan


In [16]:
extract_diff_numbers(tmp_rec['positive'], tmp_rec['negative'])

('0.36', '0.18')

In [2]:
import pandas as pd
df = pd.DataFrame(df_records)

NameError: name 'df_records' is not defined

In [43]:
# BEGIN: Filter and display records
filtered_df = df[df['negative_number'].astype(float) / df['positive_number'].astype(float) > 1]
result = filtered_df[['number', 'positive_number', 'negative_number']]
print(result.count())
# END:

number             44039
positive_number    44039
negative_number    44039
dtype: int64


In [41]:
# BEGIN: Filter and display records
filtered_df = df[df['positive_number'].astype(float) / df['negative_number'].astype(float) >3]
result = filtered_df[['number', 'positive_number', 'negative_number']]
print(result.count())

number             21
positive_number    21
negative_number    21
dtype: int64


In [20]:
unique_combinations = df[['number', 'positive_number', 'negative_number']].drop_duplicates()
print(unique_combinations)

      number positive_number  negative_number
0       27.9           29.61            23.25
1        2.4            2.07             3.30
2        3.5            3.21             3.96
3       1.26            1.17             1.46
4        1.1            0.98             1.35
...      ...             ...              ...
79790    1.1            1.23             0.88
79793  4.625            5.03             3.47
79795    120          131.88            94.87
79796  500.1           438.1           604.58
79797    3.2            3.62             1.95

[63325 rows x 3 columns]


In [21]:
unique_combinations[1000:1005]

,number,positive_number,negative_number
1085,1.91,1.63,2.35
1086,67.5,77.51,51.19
1087,8.61,9.19,7.67
1088,4.8,4.79,4.82
1089,78.1,73.04,88.02


In [22]:
unique_positive_numbers = df['positive_number'].unique()
unique_negative_numbers = df['negative_number'].unique()

print("Unique Positive Numbers:", unique_positive_numbers)
print("Unique Negative Numbers:", unique_negative_numbers)

Unique Positive Numbers: ['29.61' '2.07' '3.21' ... '1456.26' '131.88' '438.1']
Unique Negative Numbers: [  23.25    3.3     3.96 ...  175.27 1915.52  604.58]


In [23]:
len(unique_negative_numbers),len(unique_positive_numbers)

(17520, 17596)

In [35]:
zero_records = df[(df['number'] == '0') | (df['positive_number'] == '0') | (df['negative_number'] == 0)]
print(zero_records[['number','positive_number','negative_number']])

      number positive_number  negative_number
13         0            2186           2270.0
19         0            3296           3393.0
30         0            2004           2074.0
38         0            1245           1248.0
46         0            3439           3482.0
...      ...             ...              ...
79567      0            2659           2729.0
79604      0             297            357.0
79634      0            1301           1400.0
79669      0            1096           1113.0
79757      0            1671           1701.0

[4447 rows x 3 columns]


In [36]:
zero_records[123:130]

,anchor,positive,negative,number,positive_rewritten,negative_rewritten,positive_number,negative_number
2208,SHH Margin Trading -- Pci-Suntek Tech <0.0.SS>...,SHH Margin Trading -- Pci-Suntek Tech <554.SS>...,SHH Margin Trading -- Pci-Suntek Tech <-607.SS...,0,SHH Margin Trading -- Pci-Suntek Tech <554.SS>...,SHH Margin Trading -- Pci-Suntek Tech <-607.SS...,554,607.0
2259,SHH Margin Trading -- BBMG Corp <0.0.SS> Margi...,SHH Margin Trading -- BBMG Corp <2782.SS> Marg...,SHH Margin Trading -- BBMG Corp <-2788.SS> Mar...,0,SHH Margin Trading -- BBMG Corp <2782.SS> Long...,SHH Margin Trading -- BBMG Corp <-2788.SS> Lon...,2782,2788.0
2289,SHH Margin Trading -- Changhong <0.0.SS> Margi...,SHH Margin Trading -- Changhong <-2049.SS> Mar...,SHH Margin Trading -- Changhong <2126.SS> Marg...,0,SHH Margin Trading -- Changhong <-2049.SS> Lon...,SHH Margin Trading -- Changhong <2126.SS> Long...,2049,2126.0
2298,SHH Margin Trading -- CJ Publishing <0.0.SS> M...,SHH Margin Trading -- CJ Publishing <3631.SS> ...,SHH Margin Trading -- CJ Publishing <-3685.SS>...,0,SHH Margin Trading -- CJ Publishing <3631.SS> ...,SHH Margin Trading -- CJ Publishing <-3685.SS>...,3631,3685.0
2310,SHH Margin Trading -- Wuhan Jianmin <0.0.SS> M...,SHH Margin Trading -- Wuhan Jianmin <-925.SS> ...,SHH Margin Trading -- Wuhan Jianmin <946.SS> M...,0,The securities firm reported that the margin l...,The securities firm indicated that Wuhan Jianm...,925,946.0
2314,SHH Margin Trading -- Fiber Glass <0.0.SS> Mar...,SHH Margin Trading -- Fiber Glass <2600.SS> Ma...,SHH Margin Trading -- Fiber Glass <2661.SS> Ma...,0,SHH Margin Trading -- Fiber Glass <2600.SS> Ma...,SHH Margin Trading -- Fiber Glass <2661.SS> Ma...,2600,2661.0
2342,SHH Margin Trading -- Gansu Mogao <0.0.SS> Mar...,SHH Margin Trading -- Gansu Mogao <-2630.SS> M...,SHH Margin Trading -- Gansu Mogao <2685.SS> Ma...,0,SHH Margin Trading -- Gansu Mogao <-2630.SS> M...,SHH Margin Trading -- Gansu Mogao <2685.SS> Ma...,2630,2685.0


Explore training data patterns 

In [47]:
import pandas as pd
extracted_file =  "train_extracted_improved.jsonl"
df = pd.read_json(extracted_file, lines=True)

In [55]:
df.count()

anchor                103182
positive              103182
negative              103182
number                103182
positive_rewritten    103182
negative_rewritten    103182
positive_number       103182
negative_number       103182
dot_zero_fixed          6299
source                 23386
dtype: int64

In [48]:
# Count integers and floats in the 'number' column
col_name = 'negative_number'
int_count = df[col_name].apply(lambda x: isinstance(x, int)).sum()
float_count = df[col_name].apply(lambda x: isinstance(x, float)).sum()

# Find min and max values
min_value = df[col_name].min()
max_value = df[col_name].max()

print(f"Integer count: {int_count}")
print(f"Float count: {float_count}")
print(f"Minimum value: {min_value}")
print(f"Maximum value: {max_value}")

Integer count: 0
Float count: 103182
Minimum value: 0.0
Maximum value: 102768775892272.3


In [39]:
# BEGIN: Find number values that end with .00 or .0
col_name = 'number'
col_name = 'negative_number'
filtered_numbers = df[df[col_name].astype(str).str.endswith('.00') | df[col_name].astype(str).str.endswith('.0')]
print(filtered_numbers[[col_name]])
# END:

        negative_number
15              63806.0
18               4003.0
21               2592.0
45                  7.0
57               2612.0
...                 ...
103101             31.0
103103           3914.0
103112              6.0
103167           2008.0
103180           1504.0

[6891 rows x 1 columns]


In [14]:
df.columns

Index(['anchor', 'positive', 'negative', 'number', 'positive_rewritten',
       'negative_rewritten', 'positive_number', 'negative_number'],
      dtype='object')

In [17]:
import pandas as pd

# Set display options to show full content of columns
pd.set_option('display.max_colwidth', None)



In [54]:
idx = 127
df[idx:idx+1][['number','positive_number','negative_number','anchor','positive_rewritten','negative_rewritten']]

,number,positive_number,negative_number,anchor,positive_rewritten,negative_rewritten
127,0.0,1637.0,1666.0,"SHH Margin Trading -- Hongcheng Machy <0.SS> Margin Long Amount is 931 (x10000) yuan, Short Selling Volume is 0 (x10000) shares","SHH Margin Trading -- Hongcheng Machy <1637.SS> Margin Long Position totals 931 (x10000) yuan, while Short Selling Volume remains at 0 (x10000) shares","SHH Margin Trading -- Hongcheng Machy <1666.SS> Margin Long Position totals 931 (x10000) yuan, while Short Selling Volume remains at 0 (x10000) shares"


In [50]:
# BEGIN: Find number values that are greater than 1000 and end with .00 or .0
filtered_numbers = df[(df['positive_number'] > 1000) & 
                      (df['positive_number'].astype(str).str.endswith('.00') | 
                       df['positive_number'].astype(str).str.endswith('.0'))]
print(filtered_numbers[['positive_number']])
# END:

        positive_number
15              60916.0
18               3951.0
21               2516.0
57               2608.0
127              1637.0
...                 ...
103056           3513.0
103059         481949.0
103103           3905.0
103167           1923.0
103180           1452.0

[3919 rows x 1 columns]


In [12]:
filtered_numbers = df[(df['negative_number'] > 1000) & 
                      (df['negative_number'].astype(str).str.endswith('.00') | 
                       df['negative_number'].astype(str).str.endswith('.0'))]
print(filtered_numbers[['negative_number']])

       negative_number
13              1041.0
19              3589.0
38              1446.0
49              3225.0
64              1685.0
...                ...
79523           3546.0
79558           1509.0
79563           3979.0
79567           3414.0
79634           3840.0

[3382 rows x 1 columns]


In [19]:
# BEGIN: Create buckets for the 'number' column
bins = [0, 100, 1000, 10000, 100000, 1000000, float('inf')]
labels = ['< 100', '< 1000', '< 10000', '< 100000', '< 1000000', '>= 1000000']
df['number_buckets'] = pd.cut(df['number'], bins=bins, labels=labels, right=False)

# Count the number of occurrences in each bucket
bucket_counts = df['number_buckets'].value_counts().sort_index()
print(bucket_counts)
# END:

number_buckets
< 100         69178
< 1000         9277
< 10000         876
< 100000        253
< 1000000       208
>= 1000000        6
Name: count, dtype: int64


In [16]:
unique_numbers_less_than_100 = df[df['number_buckets'] == '< 1000']['number'].unique()
print(unique_numbers_less_than_100)
unique_numbers_less_than_100_int = unique_numbers_less_than_100.astype(int)
print(set(unique_numbers_less_than_100_int.tolist()))

[214.5  340.   547.5  ... 909.6  866.83 522.2 ]
{100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 2

In [21]:
# BEGIN: Create buckets for the 'positive_number' column
bins = [0, 100, 1000, 10000, 100000, 1000000, float('inf')]
labels = ['< 100', '< 1000', '< 10000', '< 100000', '< 1000000', '>= 1000000']
df['pos_number_buckets'] = pd.cut(df['positive_number'], bins=bins, labels=labels, right=False)

# Count the number of occurrences in each bucket
bucket_counts = df['pos_number_buckets'].value_counts().sort_index()
print(bucket_counts)
# END:

pos_number_buckets
< 100         64793
< 1000        10262
< 10000        4275
< 100000        250
< 1000000       212
>= 1000000        6
Name: count, dtype: int64


In [1]:
# BEGIN: Create buckets for the 'negative_number' column
bins = [0, 100, 1000, 10000, 100000, 1000000, float('inf')]
labels = ['< 100', '< 1000', '< 10000', '< 100000', '< 1000000', '>= 1000000']
df['neg_number_buckets'] = pd.cut(df['negative_number'], bins=bins, labels=labels, right=False)

# Count the number of occurrences in each bucket
bucket_counts = df['neg_number_buckets'].value_counts().sort_index()
print(bucket_counts)

NameError: name 'pd' is not defined

In [22]:
df.columns

Index(['anchor', 'positive', 'negative', 'number', 'positive_rewritten',
       'negative_rewritten', 'positive_number', 'negative_number'],
      dtype='object')

In [25]:
import numpy as np

# BEGIN: Filter records based on the specified condition
filtered_records = df[
    np.log1p(np.abs(df['number'] - df['positive_number'])) < 
    np.log1p(np.abs(df['number'] - df['negative_number']))
]
print(filtered_records)
# END:

                                                                                                                                       anchor  \
0                                                                         $27.9M City of Middletown, Connecticut Citigroup Global Markets Inc   
1                                                                       Ex-N.Y. Senate leader Bruno asks state for $2.4 million in legal fees   
2                                ADOR FONTECH LTD <ADOF.BO> - RECOMMENDED A DIVIDEND OF INR 3.5  PER SHARE FOR THE YEAR ENDED MARCH 31, 2015.   
3                                                                             Turkey's Isbank secures euro, dollar loans worth $1.26 billion    
4                                       EUROPEAN CRUDE OIL STOCKS AT 1.1 MLN BARRELS IN MAY, FLAT  FROM APRIL, DOWN 1.1 PCT Y/Y - EUROILSTOCK   
...                                                                                                                               

Claude correction
"""
Numeric Triplet Data Preparation
=================================
Two-part pipeline:
  Part 1 — Fix .00 records (strip to int / add fractional noise / leave as-is)
  Part 2 — High magnitude oversampling (oversample existing + promote by scaling)
  Part 3 — Final assembly, validation, bucket report

Assumes input JSONL with fields:
    anchor, positive_rewritten, negative_rewritten,
    number, positive_number, negative_number
"""


In [42]:
"""
Numeric Triplet Data Preparation
=================================
Two-part pipeline:
  Part 1 — Fix .00 records (strip to int / add fractional noise / leave as-is)
  Part 2 — High magnitude oversampling (oversample existing + promote by scaling)
  Part 3 — Final assembly, validation, bucket report

Assumes input JSONL with fields:
    anchor, positive_rewritten, negative_rewritten,
    number, positive_number, negative_number
"""

import re
import json
import math
import random
import logging
from pathlib import Path
from collections import defaultdict

import polars as pl

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────

INPUT_PATH  = "train_extracted.jsonl"          # your existing dataset
OUTPUT_PATH = "train_extracted_temp.jsonl"

SEED = 42
random.seed(SEED)

# Part 1 — .00 fix split
DOT_ZERO_INT_FRAC   = 0.40   # → clean integer
DOT_ZERO_FRAC_NOISE = 0.40   # → ratio-preserving fractional noise
DOT_ZERO_LEAVE      = 0.20   # → leave as-is

# Part 2 — magnitude targets (approximate counts per bucket)
MAGNITUDE_TARGETS = {
    0:  12_000,   # 1–9
    1:  12_000,   # 10–99
    2:  10_000,   # 100–999
    3:  10_000,   # 1000–9999
    4:   8_000,   # 10000–99999
    5:   6_000,   # 100000–999999
}

# Pos variation by magnitude (tighter at higher magnitudes)
POS_VARIATION_BY_MAG = {
    0: 0.20,
    1: 0.20,
    2: 0.12,
    3: 0.08,
    4: 0.05,
    5: 0.04,
}

MIN_NEG_LOG_FACTOR = 1.5   # log1p(neg_dist) must be this × log1p(pos_dist)
MAX_NEG_ATTEMPTS   = 10    # attempts to find valid negative before fallback

# Promotion scales for Part 2
PROMOTION_SCALES = [10, 100, 1_000, 10_000]


# ─────────────────────────────────────────────
# UTILITIES
# ─────────────────────────────────────────────

NUMBER_RE = re.compile(r"\d[\d,]*(?:\.\d+)?")


def get_magnitude(x: float) -> int:
    """floor(log10(abs(x))), returns -1 for x==0."""
    if x == 0:
        return -1
    return int(math.floor(math.log10(abs(x))))


def is_dot_zero(x: float) -> bool:
    """True if x has no meaningful fractional part."""
    return abs(x - round(x)) < 1e-9


def format_number(x: float, force_fraction: bool = False) -> str:
    """
    Format float cleanly.
      force_fraction=False : randomly use int repr 50% of time for .00 values
                             gives natural mix of "500" and "500.00" in original records
      force_fraction=True  : always keep 2dp — avoids introducing new .00 in generated records
    """
    if is_dot_zero(x):
        if force_fraction:
            return f"{x:.2f}"
        return str(int(round(x))) if random.random() < 0.5 else f"{x:.2f}"
    return f"{x:.2f}".rstrip("0").rstrip(".")


def replace_first_number(sentence: str, new_value: float, force_fraction: bool = False) -> str:
    """Replace the first number token in sentence with new_value."""
    formatted = format_number(new_value, force_fraction=force_fraction)
    return NUMBER_RE.sub(formatted, sentence, count=1)


def log1p_dist(a: float, b: float) -> float:
    return math.log1p(abs(a - b))


def is_valid_triplet(anchor: float, pos: float, neg: float) -> bool:
    """Triplet is valid if log-distance to positive < log-distance to negative."""
    return log1p_dist(anchor, pos) < log1p_dist(anchor, neg)


def generate_positive(anchor: float, mag: int) -> float:
    """Generate a positive number close to anchor in log space."""
    variation = POS_VARIATION_BY_MAG.get(mag, 0.10)
    for _ in range(20):
        factor = 1.0 + random.uniform(-variation, variation)
        pos = round(anchor * factor, 2)
        if pos > 0 and pos != anchor:
            return pos
    return round(anchor * 1.05, 2)


def generate_negative(anchor: float, pos: float, mag: int) -> float:
    """
    Generate a negative number that is log-space farther than positive.
    Uses magnitude shift strategy with log-distance validation.
    """
    pos_log_dist = log1p_dist(anchor, pos)

    for _ in range(MAX_NEG_ATTEMPTS):
        mag_shift = random.choice([-1, 1])
        # guard: don't shift below magnitude 0 for small anchors
        effective_shift = mag_shift if (mag + mag_shift) >= 0 else 1

        base_scale = 10 ** (mag + effective_shift)
        noise      = random.uniform(1.2, 9.5)
        neg_scale  = base_scale * noise

        neg = anchor * neg_scale if random.random() < 0.7 else anchor / neg_scale
        neg = round(abs(neg), 2)

        if neg > 0 and log1p_dist(anchor, neg) > pos_log_dist * MIN_NEG_LOG_FACTOR:
            return neg

    # fallback: multiply by large factor
    return round(anchor * (10 ** (mag + 2)) * random.uniform(1.5, 5.0), 2)


# ─────────────────────────────────────────────
# PART 1 — FIX .00 RECORDS
# ─────────────────────────────────────────────

def fix_dot_zero_record(record: dict) -> dict:
    """
    For a record where anchor ends in .00:
      - Skip zero anchors entirely (ratio logic and log-space break at 0)
      - 40% → strip all three numbers to clean integers
      - 40% → add absolute fractional offset to anchor, regenerate pos/neg fresh
      - 20% → leave as-is

    FIX 1: zero anchors skipped — was causing anchor=0 pos=1118 neg=1157
    FIX 2: use absolute offset + fresh generation instead of ratio scaling
            — was causing anchor=0.98 pos=3652 (ratio 3727x blowup)
    FIX 3: semantic proximity guard on new_pos
    """
    anchor_num = float(record["number"])

    # Bug fix: skip zero anchors entirely
    if anchor_num == 0:
        return record

    pos_num = float(record["positive_number"])
    neg_num = float(record["negative_number"])

    r = random.random()

    if r < DOT_ZERO_INT_FRAC:
        # ── Strip to clean integer ──
        # All three get integer treatment for consistent representation
        new_anchor = float(round(anchor_num))
        new_pos    = float(round(pos_num))
        new_neg    = float(round(neg_num))

    elif r < DOT_ZERO_INT_FRAC + DOT_ZERO_FRAC_NOISE:
        # ── Absolute fractional offset + fresh pos/neg generation ──
        # Add small fraction to anchor, then regenerate pos/neg around it.
        # Avoids ratio blowup bug: old code did pos = anchor * (old_pos/old_anchor)
        # which explodes when old_anchor is tiny (e.g. 0.98) and old_pos is large.
        fraction   = random.uniform(0.01, 0.98)
        new_anchor = round(anchor_num + fraction, 2)
        mag        = get_magnitude(new_anchor)

        new_pos = generate_positive(new_anchor, mag)
        new_neg = generate_negative(new_anchor, new_pos, mag)

        # Semantic proximity guard: pos must stay within 30% of anchor
        if new_anchor > 0 and abs(new_pos - new_anchor) / new_anchor > 0.30:
            return record

    else:
        # ── Leave as-is ──
        return record

    # Final triplet ordering validation
    if not is_valid_triplet(new_anchor, new_pos, new_neg):
        return record

    new_record = record.copy()
    new_record["number"]             = new_anchor
    new_record["positive_number"]    = new_pos
    new_record["negative_number"]    = new_neg
    new_record["anchor"]             = replace_first_number(record["anchor"],               new_anchor)
    new_record["positive_rewritten"] = replace_first_number(record["positive_rewritten"],   new_pos)
    new_record["negative_rewritten"] = replace_first_number(record["negative_rewritten"],   new_neg)
    new_record["dot_zero_fixed"]     = True

    return new_record


# ─────────────────────────────────────────────
# PART 2 — HIGH MAGNITUDE OVERSAMPLING
# ─────────────────────────────────────────────

def oversample_existing(
    records: list[dict],
    mag: int,
    needed: int,
) -> list[dict]:
    """
    Resample from existing records of a given magnitude with new pos/neg generation.
    Used when existing count is close to target but needs a boost.
    """
    pool = [r for r in records if get_magnitude(float(r["number"])) == mag]
    if not pool:
        return []

    new_records = []
    for _ in range(needed):
        base = random.choice(pool)
        anchor_num = float(base["number"])

        pos_num = generate_positive(anchor_num, mag)
        neg_num = generate_negative(anchor_num, pos_num, mag)

        if not is_valid_triplet(anchor_num, pos_num, neg_num):
            continue

        new_rec = base.copy()
        new_rec["number"]             = anchor_num
        new_rec["positive_number"]    = pos_num
        new_rec["negative_number"]    = neg_num
        new_rec["anchor"]             = replace_first_number(base["anchor"],               anchor_num, force_fraction=True)
        new_rec["positive_rewritten"] = replace_first_number(base["positive_rewritten"],   pos_num,    force_fraction=True)
        new_rec["negative_rewritten"] = replace_first_number(base["negative_rewritten"],   neg_num,    force_fraction=True)
        new_rec["source"] = "oversample"
        new_records.append(new_rec)

    return new_records


def promote_records(
    records: list[dict],
    target_mag: int,
    needed: int,
) -> list[dict]:
    """
    Promote low-magnitude records to target_mag by scaling all numbers.
    Naturalness is intentionally ignored — we're training numeric embeddings.

    Strategy:
      - Pick records from magnitude (target_mag - 1) or (target_mag - 2)
      - Multiply anchor/pos/neg by appropriate scale factor
      - Replace numbers in sentences
      - Regenerate pos/neg around the new scaled anchor for variety
    """
    # prefer records one magnitude below target
    source_mag = target_mag - 1
    pool = [r for r in records if get_magnitude(float(r["number"])) == source_mag]

    # fallback: go two magnitudes below
    if len(pool) < 100:
        source_mag = target_mag - 2
        pool = [r for r in records if get_magnitude(float(r["number"])) == source_mag]

    if not pool:
        log.warning(f"No source records found for promoting to magnitude {target_mag}")
        return []

    scale = 10 ** (target_mag - source_mag)
    new_records = []

    for _ in range(needed):
        base       = random.choice(pool)
        anchor_num = round(float(base["number"]) * scale, 2)
        mag        = get_magnitude(anchor_num)

        # generate fresh pos/neg around promoted anchor
        pos_num = generate_positive(anchor_num, mag)
        neg_num = generate_negative(anchor_num, pos_num, mag)

        if not is_valid_triplet(anchor_num, pos_num, neg_num):
            continue

        new_rec = base.copy()
        new_rec["number"]             = anchor_num
        new_rec["positive_number"]    = pos_num
        new_rec["negative_number"]    = neg_num
        new_rec["anchor"]             = replace_first_number(base["anchor"],               anchor_num, force_fraction=True)
        new_rec["positive_rewritten"] = replace_first_number(base["positive_rewritten"],   pos_num,    force_fraction=True)
        new_rec["negative_rewritten"] = replace_first_number(base["negative_rewritten"],   neg_num,    force_fraction=True)
        new_rec["source"]             = f"promoted_x{scale}"
        new_records.append(new_rec)

    return new_records


# ─────────────────────────────────────────────
# PART 3 — VALIDATION & BUCKET REPORT
# ─────────────────────────────────────────────

def validate_and_report(records: list[dict]) -> list[dict]:
    """
    Final pass:
      - Remove any triplets that violate log-distance ordering
      - Print bucket distribution before and after
    """
    valid   = []
    invalid = 0

    bucket_counts = defaultdict(int)

    for rec in records:
        a = float(rec["number"])
        p = float(rec["positive_number"])
        n = float(rec["negative_number"])

        if not is_valid_triplet(a, p, n):
            invalid += 1
            continue

        valid.append(rec)
        mag = get_magnitude(a)
        bucket_counts[mag] += 1

    log.info(f"Validation: {len(valid)} valid, {invalid} removed")
    log.info("─── Final Bucket Distribution ───")

    total = len(valid)
    for mag in sorted(bucket_counts):
        count = bucket_counts[mag]
        pct   = 100 * count / total if total > 0 else 0
        upper = 10 ** (mag + 1)
        label = f"< {upper:>10,}"
        bar   = "█" * int(pct / 2)
        log.info(f"  mag={mag} {label} : {count:6d} ({pct:5.1f}%) {bar}")

    return valid


# ─────────────────────────────────────────────
# MAIN PIPELINE
# ─────────────────────────────────────────────

def run_pipeline(input_path: str, output_path: str):

    # ── Load ──
    log.info(f"Loading {input_path}")
    records = []
    with open(input_path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    log.info(f"Loaded {len(records):,} records")

    # ── Bucket report BEFORE ──
    log.info("─── Input Bucket Distribution ───")
    before_buckets = defaultdict(int)
    for r in records:
        before_buckets[get_magnitude(float(r["number"]))] += 1
    for mag in sorted(before_buckets):
        log.info(f"  mag={mag} : {before_buckets[mag]:,}")

    # ────────────────────────────────────────
    # PART 1: Fix .00 records
    # ────────────────────────────────────────
    log.info("Part 1: Fixing .00 records ...")
    fixed_records = []
    dot_zero_count = 0

    for rec in records:
        anchor_num = float(rec["number"])
        if is_dot_zero(anchor_num):
            dot_zero_count += 1
            rec = fix_dot_zero_record(rec)
        fixed_records.append(rec)

    log.info(f"  .00 records processed: {dot_zero_count:,}")

    # ────────────────────────────────────────
    # PART 2: High magnitude oversampling
    # ────────────────────────────────────────
    log.info("Part 2: High magnitude oversampling ...")
    all_records = fixed_records.copy()

    current_counts = defaultdict(int)
    for r in all_records:
        current_counts[get_magnitude(float(r["number"]))] += 1

    for mag, target in MAGNITUDE_TARGETS.items():
        current = current_counts.get(mag, 0)
        needed  = max(0, target - current)

        if needed == 0:
            log.info(f"  mag={mag}: already at {current:,}, no oversampling needed")
            continue

        log.info(f"  mag={mag}: current={current:,}, target={target:,}, generating {needed:,}")

        if current >= needed // 2:
            # enough existing records → oversample from existing
            new_recs = oversample_existing(all_records, mag, needed)
        else:
            # too few existing → promote from lower magnitudes
            new_recs = promote_records(all_records, mag, needed)

        log.info(f"    → generated {len(new_recs):,} new records for mag={mag}")
        all_records.extend(new_recs)

        # update counts for subsequent magnitude iterations
        current_counts[mag] += len(new_recs)

    # ────────────────────────────────────────
    # PART 3: Validate & report
    # ────────────────────────────────────────
    log.info("Part 3: Validating and reporting ...")
    final_records = validate_and_report(all_records)

    # shuffle before writing
    random.shuffle(final_records)

    # ── Write output ──
    log.info(f"Writing {len(final_records):,} records to {output_path}")
    with open(output_path, "w") as f:
        for rec in final_records:
            f.write(json.dumps(rec) + "\n")

    log.info("Done.")
    return final_records



final = run_pipeline(INPUT_PATH, OUTPUT_PATH)
log.info(f"Final dataset size: {len(final):,}")

2026-02-23 23:45:38,015 | INFO | Loading train_extracted.jsonl
2026-02-23 23:45:38,364 | INFO | Loaded 79,798 records
2026-02-23 23:45:38,365 | INFO | ─── Input Bucket Distribution ───
2026-02-23 23:45:38,393 | INFO |   mag=-4 : 3
2026-02-23 23:45:38,394 | INFO |   mag=-3 : 79
2026-02-23 23:45:38,394 | INFO |   mag=-2 : 2,559
2026-02-23 23:45:38,394 | INFO |   mag=-1 : 16,408
2026-02-23 23:45:38,395 | INFO |   mag=0 : 30,851
2026-02-23 23:45:38,395 | INFO |   mag=1 : 19,278
2026-02-23 23:45:38,395 | INFO |   mag=2 : 9,277
2026-02-23 23:45:38,396 | INFO |   mag=3 : 876
2026-02-23 23:45:38,396 | INFO |   mag=4 : 253
2026-02-23 23:45:38,396 | INFO |   mag=5 : 208
2026-02-23 23:45:38,396 | INFO |   mag=6 : 5
2026-02-23 23:45:38,397 | INFO |   mag=11 : 1
2026-02-23 23:45:38,397 | INFO | Part 1: Fixing .00 records ...
2026-02-23 23:45:38,447 | INFO |   .00 records processed: 9,686
2026-02-23 23:45:38,447 | INFO | Part 2: High magnitude oversampling ...
2026-02-23 23:45:38,524 | INFO |   mag=

In [43]:
"""
Strip .00 from numbers and sentences in JSONL triplet dataset.

Usage:
    python strip_dot_zero.py --input triplets_improved.jsonl --output triplets_fixed.jsonl
"""

import re
import json
import argparse
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)

DOT_ZERO_RE = re.compile(r'\.0+\b')


def strip_dot_zero_num(x) -> float | int:
    """Return int if value is whole number, else original float."""
    f = float(x)
    return int(f) if f == int(f) else f


def strip_dot_zero_str(s: str) -> str:
    """Remove .00 / .000 etc from number tokens in a sentence."""
    return DOT_ZERO_RE.sub("", s)


def fix_record(rec: dict) -> dict:
    fixed = rec.copy()
    for field in ("number", "positive_number", "negative_number"):
        if field in fixed:
            fixed[field] = strip_dot_zero_num(fixed[field])
    for field in ("anchor", "positive_rewritten", "negative_rewritten"):
        if field in fixed:
            fixed[field] = strip_dot_zero_str(fixed[field])
    return fixed


def run(input_path: str, output_path: str):
    records = []
    with open(input_path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    log.info(f"Loaded {len(records):,} records")

    fixed = [fix_record(r) for r in records]

    with open(output_path, "w") as f:
        for rec in fixed:
            f.write(json.dumps(rec) + "\n")
    log.info(f"Written {len(fixed):,} records to {output_path}")




run("train_extracted_temp.jsonl", "train_extracted_improved.jsonl")

2026-02-23 23:45:57,792 | INFO | Loaded 103,182 records
2026-02-23 23:45:58,489 | INFO | Written 103,182 records to train_extracted_improved.jsonl


In [44]:
"""
Triplet Dataset Validation
===========================
Validates the output of data_preparation.py across 7 checks:

  1. Triplet ordering integrity
  2. Bucket distribution (before vs after)
  3. .00 fix verification
  4. Sentence-number consistency
  5. Promoted records sanity check
  6. Duplicate detection
  7. Field completeness

Usage:
    python validate_dataset.py \
        --original triplets.jsonl \
        --improved triplets_improved.jsonl
"""

import re
import json
import math
import argparse
import logging
from collections import defaultdict, Counter
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────

REQUIRED_FIELDS = [
    "anchor", "positive_rewritten", "negative_rewritten",
    "number", "positive_number", "negative_number",
]

NUMBER_RE = re.compile(r"\d[\d,]*(?:\.\d+)?")

# How close the extracted sentence number must be to the field value
# to count as consistent (handles floating point formatting differences)
NUMBER_MATCH_TOL = 0.02   # 2% relative tolerance


# ─────────────────────────────────────────────
# UTILITIES
# ─────────────────────────────────────────────

def load_jsonl(path: str) -> list[dict]:
    records = []
    with open(path) as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                log.warning(f"  Line {i}: JSON parse error — {e}")
    return records


def get_magnitude(x: float) -> int:
    if x == 0:
        return -1
    return int(math.floor(math.log10(abs(x))))


def log1p_dist(a: float, b: float) -> float:
    return math.log1p(abs(a - b))


def is_dot_zero(x: float) -> bool:
    return abs(x - round(x)) < 1e-9


def extract_first_number(sentence: str) -> float | None:
    """Extract the first number from a sentence, removing commas."""
    m = NUMBER_RE.search(sentence)
    if m:
        try:
            return float(m.group().replace(",", ""))
        except ValueError:
            return None
    return None


def numbers_match(extracted: float, expected: float, tol: float = NUMBER_MATCH_TOL) -> bool:
    """Check if extracted number is within tol relative tolerance of expected."""
    if expected == 0:
        return abs(extracted) < 1e-6
    return abs(extracted - expected) / abs(expected) <= tol


def section(title: str):
    log.info("")
    log.info("=" * 60)
    log.info(f"  {title}")
    log.info("=" * 60)


def result(label: str, passed: bool, detail: str = ""):
    status = "✅ PASS" if passed else "❌ FAIL"
    msg = f"  {status} — {label}"
    if detail:
        msg += f" | {detail}"
    log.info(msg)


# ─────────────────────────────────────────────
# CHECK 1: Triplet ordering integrity
# ─────────────────────────────────────────────

def check_triplet_ordering(records: list[dict]) -> bool:
    section("CHECK 1: Triplet Ordering Integrity")
    violations = []

    for i, rec in enumerate(records):
        try:
            a = float(rec["number"])
            p = float(rec["positive_number"])
            n = float(rec["negative_number"])
        except (KeyError, ValueError, TypeError):
            continue

        if log1p_dist(a, p) >= log1p_dist(a, n):
            violations.append({
                "idx": i,
                "anchor": a,
                "pos": p,
                "neg": n,
                "pos_log_dist": round(log1p_dist(a, p), 4),
                "neg_log_dist": round(log1p_dist(a, n), 4),
            })

    passed = len(violations) == 0
    result(
        "All triplets satisfy log1p(|a-p|) < log1p(|a-n|)",
        passed,
        f"{len(violations)} violations out of {len(records):,}"
    )

    if violations:
        log.info("  First 5 violations:")
        for v in violations[:5]:
            log.info(f"    idx={v['idx']} anchor={v['anchor']} "
                     f"pos={v['pos']} (log_dist={v['pos_log_dist']}) "
                     f"neg={v['neg']} (log_dist={v['neg_log_dist']})")

    return passed


# ─────────────────────────────────────────────
# CHECK 2: Bucket distribution
# ─────────────────────────────────────────────

def check_bucket_distribution(
    original: list[dict],
    improved: list[dict],
) -> bool:
    section("CHECK 2: Bucket Distribution (Before vs After)")

    def get_buckets(records):
        counts = defaultdict(int)
        for r in records:
            try:
                counts[get_magnitude(float(r["number"]))] += 1
            except (KeyError, ValueError):
                pass
        return counts

    before = get_buckets(original)
    after  = get_buckets(improved)

    all_mags = sorted(set(list(before.keys()) + list(after.keys())))
    total_before = sum(before.values())
    total_after  = sum(after.values())

    log.info(f"  {'mag':<6} {'upper':<12} {'before':>8} {'before%':>8} {'after':>8} {'after%':>8} {'delta':>8}")
    log.info(f"  {'-'*6} {'-'*12} {'-'*8} {'-'*8} {'-'*8} {'-'*8} {'-'*8}")

    high_mag_improved = True
    for mag in all_mags:
        b = before.get(mag, 0)
        a = after.get(mag, 0)
        b_pct = 100 * b / total_before if total_before > 0 else 0
        a_pct = 100 * a / total_after  if total_after  > 0 else 0
        delta = a - b
        upper = f"< {10 ** (mag+1):>10,}" if mag >= 0 else "  zero/neg"
        log.info(f"  {mag:<6} {upper:<12} {b:>8,} {b_pct:>7.1f}% {a:>8,} {a_pct:>7.1f}% {delta:>+8,}")

        # flag if high magnitude buckets didn't improve
        if mag >= 2 and a_pct < 5.0:
            high_mag_improved = False

    log.info(f"\n  Total: {total_before:,} → {total_after:,} "
             f"(+{total_after - total_before:,})")

    result("High magnitude buckets (mag>=2) each have >= 5% share", high_mag_improved)
    return high_mag_improved


# ─────────────────────────────────────────────
# CHECK 3: .00 fix verification
# ─────────────────────────────────────────────

def check_dot_zero_fix(
    original: list[dict],
    improved: list[dict],
) -> bool:
    section("CHECK 3: .00 Fix Verification")

    orig_dot_zero  = sum(1 for r in original if is_dot_zero(float(r.get("number", 1))))
    impr_dot_zero  = sum(1 for r in improved if is_dot_zero(float(r.get("number", 1))))
    impr_int       = sum(1 for r in improved
                         if isinstance(r.get("number"), (int, float))
                         and is_dot_zero(float(r["number"]))
                         and float(r["number"]) == int(float(r["number"])))

    # check fixed records (tagged with dot_zero_fixed)
    fixed_records  = [r for r in improved if r.get("dot_zero_fixed")]
    fixed_count    = len(fixed_records)

    log.info(f"  Original .00 anchors   : {orig_dot_zero:,}")
    log.info(f"  Improved .00 anchors   : {impr_dot_zero:,}")
    log.info(f"  Records tagged fixed   : {fixed_count:,}")

    # spot check: sample 5 fixed records
    if fixed_records:
        log.info("  Sample of fixed records:")
        for r in fixed_records[:5]:
            log.info(f"    anchor={r['number']}  pos={r['positive_number']}  neg={r['negative_number']}")

    reduction = orig_dot_zero - impr_dot_zero
    passed = reduction > 0 or orig_dot_zero == 0
    result(
        ".00 anchor count reduced",
        passed,
        f"{orig_dot_zero:,} → {impr_dot_zero:,} (reduced by {reduction:,})"
    )
    return passed


# ─────────────────────────────────────────────
# CHECK 4: Sentence-number consistency
# ─────────────────────────────────────────────

def check_sentence_consistency(records: list[dict], sample_size: int = 5000) -> bool:
    section("CHECK 4: Sentence-Number Consistency")

    # sample for speed on large datasets
    sample = records if len(records) <= sample_size else \
             [records[i] for i in range(0, len(records), max(1, len(records) // sample_size))]

    anchor_mismatches   = []
    pos_mismatches      = []
    neg_mismatches      = []

    for i, rec in enumerate(sample):
        try:
            a_val = float(rec["number"])
            p_val = float(rec["positive_number"])
            n_val = float(rec["negative_number"])
        except (KeyError, ValueError):
            continue

        # check anchor sentence
        a_extracted = extract_first_number(rec.get("anchor", ""))
        if a_extracted is not None and not numbers_match(a_extracted, a_val):
            anchor_mismatches.append({
                "idx": i, "expected": a_val,
                "extracted": a_extracted, "sentence": rec["anchor"][:80]
            })

        # check positive sentence
        p_extracted = extract_first_number(rec.get("positive_rewritten", ""))
        if p_extracted is not None and not numbers_match(p_extracted, p_val):
            pos_mismatches.append({
                "idx": i, "expected": p_val,
                "extracted": p_extracted, "sentence": rec["positive_rewritten"][:80]
            })

        # check negative sentence
        n_extracted = extract_first_number(rec.get("negative_rewritten", ""))
        if n_extracted is not None and not numbers_match(n_extracted, n_val):
            neg_mismatches.append({
                "idx": i, "expected": n_val,
                "extracted": n_extracted, "sentence": rec["negative_rewritten"][:80]
            })

    total = len(sample)
    a_pct = 100 * len(anchor_mismatches) / total
    p_pct = 100 * len(pos_mismatches)    / total
    n_pct = 100 * len(neg_mismatches)    / total

    log.info(f"  Sampled {total:,} records")
    log.info(f"  Anchor mismatches   : {len(anchor_mismatches):,} ({a_pct:.1f}%)")
    log.info(f"  Positive mismatches : {len(pos_mismatches):,}    ({p_pct:.1f}%)")
    log.info(f"  Negative mismatches : {len(neg_mismatches):,}    ({n_pct:.1f}%)")

    if anchor_mismatches:
        log.info("  Sample anchor mismatches:")
        for m in anchor_mismatches[:3]:
            log.info(f"    expected={m['expected']} extracted={m['extracted']} | {m['sentence']}")

    if pos_mismatches:
        log.info("  Sample positive mismatches:")
        for m in pos_mismatches[:3]:
            log.info(f"    expected={m['expected']} extracted={m['extracted']} | {m['sentence']}")

    passed = a_pct < 5.0 and p_pct < 5.0 and n_pct < 5.0
    result(
        "Sentence numbers match field values (< 5% mismatch)",
        passed,
        f"anchor={a_pct:.1f}% pos={p_pct:.1f}% neg={n_pct:.1f}%"
    )
    return passed


# ─────────────────────────────────────────────
# CHECK 5: Promoted records sanity check
# ─────────────────────────────────────────────

def check_promoted_records(records: list[dict]) -> bool:
    section("CHECK 5: Promoted Records Sanity Check")

    promoted = [r for r in records if str(r.get("source", "")).startswith("promoted")]
    if not promoted:
        log.info("  No promoted records found — skipping")
        result("Promoted records check", True, "no promoted records")
        return True

    log.info(f"  Total promoted records: {len(promoted):,}")

    # group by scale
    by_scale = defaultdict(list)
    for r in promoted:
        by_scale[r.get("source", "unknown")].append(r)

    issues = []
    for source, recs in by_scale.items():
        log.info(f"\n  Source: {source} ({len(recs):,} records)")
        log.info("  Sample:")
        for r in recs[:3]:
            a = float(r["number"])
            p = float(r["positive_number"])
            n = float(r["negative_number"])
            log.info(f"    anchor={a}  pos={p}  neg={n}  "
                     f"mag={get_magnitude(a)}  "
                     f"pos_log_dist={log1p_dist(a,p):.3f}  "
                     f"neg_log_dist={log1p_dist(a,n):.3f}")

        # check that all promoted records are in the expected magnitude range
        for r in recs:
            a = float(r["number"])
            if get_magnitude(a) < 2:
                issues.append(f"Promoted record has low magnitude: anchor={a}")

    passed = len(issues) == 0
    result(
        "All promoted records have magnitude >= 2",
        passed,
        f"{len(issues)} issues found" if issues else "all good"
    )

    if issues:
        for issue in issues[:5]:
            log.info(f"    {issue}")

    return passed


# ─────────────────────────────────────────────
# CHECK 6: Duplicate detection
# ─────────────────────────────────────────────

def check_duplicates(records: list[dict]) -> bool:
    section("CHECK 6: Duplicate Detection")

    # check exact anchor sentence duplicates
    anchor_counter = Counter(r.get("anchor", "") for r in records)
    exact_dups     = {k: v for k, v in anchor_counter.items() if v > 1}

    # check (anchor, pos, neg) tuple duplicates
    triplet_counter = Counter(
        (r.get("anchor", ""), r.get("positive_rewritten", ""), r.get("negative_rewritten", ""))
        for r in records
    )
    full_dups = {k: v for k, v in triplet_counter.items() if v > 1}

    total_excess_anchor  = sum(v - 1 for v in exact_dups.values())
    total_excess_triplet = sum(v - 1 for v in full_dups.values())

    dup_pct = 100 * total_excess_triplet / len(records) if records else 0

    log.info(f"  Unique anchor sentences     : {len(anchor_counter):,} "
             f"({len(exact_dups):,} have duplicates, {total_excess_anchor:,} excess)")
    log.info(f"  Unique full triplets        : {len(triplet_counter):,} "
             f"({len(full_dups):,} have duplicates, {total_excess_triplet:,} excess)")
    log.info(f"  Duplicate triplet rate      : {dup_pct:.2f}%")

    if full_dups:
        log.info("  Most duplicated triplets (top 5):")
        for triplet, count in sorted(full_dups.items(), key=lambda x: -x[1])[:5]:
            log.info(f"    count={count} | anchor='{triplet[0][:60]}'")

    passed = dup_pct < 10.0
    result(
        "Duplicate full triplets < 10%",
        passed,
        f"{dup_pct:.2f}% duplicates"
    )
    return passed


# ─────────────────────────────────────────────
# CHECK 7: Field completeness
# ─────────────────────────────────────────────

def check_field_completeness(records: list[dict]) -> bool:
    section("CHECK 7: Field Completeness")

    missing_counts  = defaultdict(int)
    empty_counts    = defaultdict(int)
    type_errors     = defaultdict(int)

    for rec in records:
        for field in REQUIRED_FIELDS:
            val = rec.get(field)

            if val is None:
                missing_counts[field] += 1
            elif isinstance(val, str) and val.strip() == "":
                empty_counts[field] += 1
            elif field in ("number", "positive_number", "negative_number"):
                try:
                    float(val)
                except (ValueError, TypeError):
                    type_errors[field] += 1

    total = len(records)
    all_passed = True

    for field in REQUIRED_FIELDS:
        missing = missing_counts[field]
        empty   = empty_counts[field]
        terror  = type_errors.get(field, 0)
        issues  = missing + empty + terror
        pct     = 100 * issues / total if total > 0 else 0
        passed  = issues == 0

        if not passed:
            all_passed = False

        result(
            f"Field '{field}' complete",
            passed,
            f"missing={missing} empty={empty} type_errors={terror} ({pct:.2f}%)"
        )

    return all_passed


# ─────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────

def print_summary(results: dict[str, bool]):
    section("VALIDATION SUMMARY")
    all_passed = all(results.values())

    for check, passed in results.items():
        status = "✅" if passed else "❌"
        log.info(f"  {status}  {check}")

    log.info("")
    if all_passed:
        log.info("  🎉 All checks passed — dataset is ready for training")
    else:
        failed = [k for k, v in results.items() if not v]
        log.info(f"  ⚠️  {len(failed)} check(s) failed — review before training")


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────





In [49]:



log.info(f"Loading original : ")
original = load_jsonl("train_extracted_temp.jsonl")
log.info(f"  {len(original):,} records")

log.info(f"Loading improved ")
improved = load_jsonl("train_extracted_improved.jsonl")
log.info(f"  {len(improved):,} records")

results = {}

results["1. Triplet ordering integrity"]   = check_triplet_ordering(improved)
results["2. Bucket distribution improved"] = check_bucket_distribution(original, improved)
results["3. .00 fix applied"]              = check_dot_zero_fix(original, improved)
results["4. Sentence-number consistency"]  = check_sentence_consistency(improved)
results["5. Promoted records sanity"]      = check_promoted_records(improved)
results["6. Duplicate rate < 10%"]         = check_duplicates(improved)
results["7. Field completeness"]           = check_field_completeness(improved)

print_summary(results)

2026-02-23 23:52:01,042 | INFO | Loading original : 
2026-02-23 23:52:04,013 | INFO |   103,182 records
2026-02-23 23:52:04,019 | INFO | Loading improved 
2026-02-23 23:52:05,285 | INFO |   103,182 records
2026-02-23 23:52:05,290 | INFO | 
2026-02-23 23:52:05,290 | INFO | ============================================================
2026-02-23 23:52:05,291 | INFO |   CHECK 1: Triplet Ordering Integrity
2026-02-23 23:52:05,292 | INFO | ============================================================
2026-02-23 23:52:05,401 | INFO |   ✅ PASS — All triplets satisfy log1p(|a-p|) < log1p(|a-n|) | 0 violations out of 103,182
2026-02-23 23:52:05,401 | INFO | 
2026-02-23 23:52:05,401 | INFO | ============================================================
2026-02-23 23:52:05,401 | INFO |   CHECK 2: Bucket Distribution (Before vs After)
2026-02-23 23:52:05,402 | INFO | ============================================================
2026-02-23 23:52:05,523 | INFO |   mag    upper          before  before%  

Duplicate and small number correction

In [56]:
"""
Deduplication + Minimum Log-Distance Filter
=============================================
Two-pass cleanup on triplet JSONL:

  Pass 1 — Deduplicate on (number, positive_number, negative_number)
            Keep at most MAX_DUPLICATES instances per numeric triple.
            Preserves sentence diversity within a triple up to the cap.

  Pass 2 — Minimum log-distance filter
            Remove triplets where log1p(|anchor - pos|) < MIN_POS_LOG_DIST
            These are cases where anchor and positive are so close that
            the distinction is below the model's learning resolution,
            especially problematic when tokenization gives no signal
            (e.g. anchor=1.5, pos=1.49 → tokens [1,.,5] vs [1,.,49])

Usage:
    python dedup_filter.py \
        --input  splits/train.jsonl \
        --output splits/train_clean.jsonl \
        [--max-dupes 3] \
        [--min-pos-log-dist 0.05]

Run separately on train, val, test.
Only train needs aggressive dedup — val/test can use higher max-dupes
since you want eval to be representative not deduplicated.
"""

import json
import math
import logging
import argparse
from collections import defaultdict

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)


# ─────────────────────────────────────────────
# CONFIG DEFAULTS
# ─────────────────────────────────────────────

MAX_DUPLICATES    = 3      # max instances of same (anchor, pos, neg) numeric triple
MIN_POS_LOG_DIST  = 0.05   # log1p(|anchor - pos|) must exceed this
                            # e^0.05 - 1 ≈ 0.051 absolute difference minimum
                            # filters cases like anchor=1.5, pos=1.49 (diff=0.01)


# ─────────────────────────────────────────────
# UTILITIES
# ─────────────────────────────────────────────

def log1p_dist(a: float, b: float) -> float:
    return math.log1p(abs(a - b))


def is_valid_triplet(a: float, p: float, n: float) -> bool:
    return log1p_dist(a, p) < log1p_dist(a, n)


def numeric_key(rec: dict) -> tuple:
    """
    Canonical key for deduplication.
    Rounds to 4dp to treat near-identical floats as same triple
    (e.g. 1.5000001 and 1.5 from float precision artifacts).
    """
    try:
        a = round(float(rec["number"]),          4)
        p = round(float(rec["positive_number"]), 4)
        n = round(float(rec["negative_number"]), 4)
        return (a, p, n)
    except (KeyError, ValueError, TypeError):
        return None


# ─────────────────────────────────────────────
# PASS 1 — DEDUPLICATION
# ─────────────────────────────────────────────

def deduplicate(records: list[dict], max_dupes: int) -> tuple[list[dict], dict]:
    """
    Keep at most max_dupes records per (anchor, pos, neg) numeric triple.
    Within each triple, shuffle first so the kept instances are random
    rather than always the first ones encountered (avoids source bias).
    """
    import random
    random.seed(42)

    # Group by numeric key
    groups = defaultdict(list)
    no_key = []
    for rec in records:
        key = numeric_key(rec)
        if key is None:
            no_key.append(rec)
        else:
            groups[key].append(rec)

    kept        = []
    removed     = 0
    over_cap    = 0   # triples that exceeded the cap

    for key, group in groups.items():
        random.shuffle(group)
        if len(group) > max_dupes:
            over_cap += 1
            removed  += len(group) - max_dupes
            kept.extend(group[:max_dupes])
        else:
            kept.extend(group)

    kept.extend(no_key)

    stats = {
        "before":      len(records),
        "after":       len(kept),
        "removed":     removed,
        "over_cap":    over_cap,
        "unique_triples": len(groups),
    }
    return kept, stats


# ─────────────────────────────────────────────
# PASS 2 — MINIMUM LOG-DISTANCE FILTER
# ─────────────────────────────────────────────

def filter_min_distance(records: list[dict], min_pos_log_dist: float) -> tuple[list[dict], dict]:
    """
    Remove triplets where positive is too close to anchor in log space.

    Two sub-cases filtered:
      a) pos too close to anchor: log1p(|a-p|) < min_pos_log_dist
         → model has no resolution to learn this distinction
         → tokenization gives no help (e.g. 1.5 vs 1.49)

      b) anchor == positive numerically (distance = 0)
         → degenerate triplet, should not exist but guard anyway
    """
    kept      = []
    removed_close    = 0
    removed_equal    = 0
    removed_invalid  = 0

    for rec in records:
        try:
            a = float(rec["number"])
            p = float(rec["positive_number"])
            n = float(rec["negative_number"])
        except (KeyError, ValueError, TypeError):
            kept.append(rec)
            continue

        # Degenerate: anchor == positive
        if a == p:
            removed_equal += 1
            continue

        # Positive too close to anchor
        if log1p_dist(a, p) < min_pos_log_dist:
            removed_close += 1
            continue

        # Final validity check (should already be valid but guard)
        if not is_valid_triplet(a, p, n):
            removed_invalid += 1
            continue

        kept.append(rec)

    stats = {
        "before":           len(records),
        "after":            len(kept),
        "removed_close":    removed_close,
        "removed_equal":    removed_equal,
        "removed_invalid":  removed_invalid,
    }
    return kept, stats


# ─────────────────────────────────────────────
# REPORTING
# ─────────────────────────────────────────────

def bucket_report(records: list[dict], label: str):
    """Print magnitude bucket distribution."""
    counts = defaultdict(int)
    for r in records:
        try:
            x = float(r["number"])
            if x == 0:
                mag = -1
            else:
                mag = int(math.floor(math.log10(abs(x))))
            counts[mag] += 1
        except (KeyError, ValueError):
            pass

    total = sum(counts.values())
    log.info(f"  {label} bucket distribution:")
    for mag in sorted(counts):
        count = counts[mag]
        pct   = 100 * count / total if total > 0 else 0
        upper = f"< {10**(mag+1):>10,}" if mag >= 0 else "     zero/neg"
        bar   = "█" * int(pct / 2)
        log.info(f"    mag={mag} {upper} : {count:6,} ({pct:5.1f}%) {bar}")


def duplicate_report(records: list[dict], top_n: int = 10):
    """Show top remaining duplicates after dedup."""
    counts = defaultdict(int)
    for rec in records:
        key = numeric_key(rec)
        if key:
            counts[key] += 1

    dupes = {k: v for k, v in counts.items() if v > 1}
    if not dupes:
        log.info("  No duplicates remaining")
        return

    log.info(f"  Remaining duplicates (top {top_n}):")
    for key, count in sorted(dupes.items(), key=lambda x: -x[1])[:top_n]:
        log.info(f"    {key} → {count}×")


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

def run(input_path: str, output_path: str, max_dupes: int, min_pos_log_dist: float):

    # ── Load ──
    log.info(f"Loading {input_path}")
    records = []
    with open(input_path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    log.info(f"  Loaded {len(records):,} records")

    bucket_report(records, "Before")

    # ── Pass 1: Deduplicate ──
    log.info("")
    log.info(f"Pass 1: Deduplication (max {max_dupes} per numeric triple)")
    deduped, dedup_stats = deduplicate(records, max_dupes)

    log.info(f"  Unique numeric triples  : {dedup_stats['unique_triples']:,}")
    log.info(f"  Triples over cap        : {dedup_stats['over_cap']:,}")
    log.info(f"  Records removed         : {dedup_stats['removed']:,}")
    log.info(f"  Records remaining       : {dedup_stats['after']:,}")

    duplicate_report(deduped)

    # ── Pass 2: Minimum distance filter ──
    log.info("")
    log.info(f"Pass 2: Minimum log-distance filter (min_pos_log_dist={min_pos_log_dist})")
    log.info(f"  Absolute diff threshold : ~{math.expm1(min_pos_log_dist):.4f}")

    filtered, filter_stats = filter_min_distance(deduped, min_pos_log_dist)

    log.info(f"  Removed (too close)     : {filter_stats['removed_close']:,}")
    log.info(f"  Removed (equal)         : {filter_stats['removed_equal']:,}")
    log.info(f"  Removed (invalid)       : {filter_stats['removed_invalid']:,}")
    log.info(f"  Records remaining       : {filter_stats['after']:,}")

    # ── Final report ──
    log.info("")
    log.info("─── Final Summary ───")
    log.info(f"  Input   : {len(records):,}")
    log.info(f"  Output  : {len(filtered):,}")
    log.info(f"  Removed : {len(records) - len(filtered):,} "
             f"({100*(len(records)-len(filtered))/len(records):.1f}%)")

    bucket_report(filtered, "After")

    # ── Write ──
    with open(output_path, "w") as f:
        for rec in filtered:
            f.write(json.dumps(rec) + "\n")
    log.info(f"  Written to {output_path}")
    log.info("Done.")


# ─────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────


run(
    input_path       = "train_extracted_improved.jsonl",
    output_path      = "train_extracted_improved_nodup.jsonl",
    max_dupes        = 3,
    min_pos_log_dist = 0.07,
)

2026-02-25 07:34:47,191 | INFO | Loading train_extracted_improved.jsonl
2026-02-25 07:34:48,948 | INFO |   Loaded 103,182 records
2026-02-25 07:34:48,981 | INFO |   Before bucket distribution:
2026-02-25 07:34:48,983 | INFO |     mag=-4      zero/neg :      3 (  0.0%) 
2026-02-25 07:34:48,984 | INFO |     mag=-3      zero/neg :     79 (  0.1%) 
2026-02-25 07:34:48,984 | INFO |     mag=-2      zero/neg :  2,559 (  2.5%) █
2026-02-25 07:34:48,985 | INFO |     mag=-1      zero/neg : 16,408 ( 15.9%) ███████
2026-02-25 07:34:48,985 | INFO |     mag=0 <         10 : 30,849 ( 29.9%) ██████████████
2026-02-25 07:34:48,985 | INFO |     mag=1 <        100 : 19,278 ( 18.7%) █████████
2026-02-25 07:34:48,986 | INFO |     mag=2 <      1,000 : 10,000 (  9.7%) ████
2026-02-25 07:34:48,986 | INFO |     mag=3 <     10,000 : 10,000 (  9.7%) ████
2026-02-25 07:34:48,987 | INFO |     mag=4 <    100,000 :  8,000 (  7.8%) ███
2026-02-25 07:34:48,987 | INFO |     mag=5 <  1,000,000 :  6,000 (  5.8%) ██
2026-